In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from scipy.special import softmax
from tqdm import tqdm

# -------------------------------
# Load your training data
# -------------------------------
df_train = pd.read_csv('/home/ankur/Desktop/dl_1000/1000_ml_projects/data/jigsaw-kaggle/jigsaw-agile-community-rules/train.csv')

df =df_train

/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# -------------------------------
# Load Qwen2.5-0.5B INT4 model
# -------------------------------
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    load_in_4bit=True,       # INT4 GPTQ
    torch_dtype=torch.float16,
    trust_remote_code=True
)

`torch_dtype` is deprecated! Use `dtype` instead!
2025-10-06 13:14:18.935319: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 13:14:18.964713: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 13:14:19.959264: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The `load_in_4bit` and `load_in_8bit` arg

In [5]:
SYS_PROMPT = """
You are given a comment on Reddit. Your task is to classify if it violates the given rule. Only respond Yes or No.
"""
prompts = []

for i, row in df.iterrows():
    text = f"""
r/{row.subreddit}
Rule: {row.rule}

1) {row.positive_example_1}
Violation: Yes

2) {row.positive_example_2}
Violation: Yes

3) {row.negative_example_1}
Violation: No

4) {row.negative_example_2}
Violation: No

5) {row.body}
Answer:
"""
    full_prompt = SYS_PROMPT + "\n" + text
    prompts.append(full_prompt)

df["prompts"] = prompts

# -------------------------------
# Batched prediction
# -------------------------------
batch_size = 8  # adjust depending on GPU memory
predictions = []
logits_yes_no = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True)
    # .to("cuda")

    # with torch.no_grad():
    outputs = model(**inputs)
        # logits for last token in sequence
    batch_logits = outputs.logits[:, -1, :]

    for j in range(batch_logits.shape[0]):
        logits = batch_logits[j]

        # Yes/No token IDs
        yes_id = tokenizer("Yes", add_special_tokens=False).input_ids[0]
        no_id = tokenizer("No", add_special_tokens=False).input_ids[0]

        # select logits
        selected_logits = torch.tensor([logits[yes_id].item(), logits[no_id].item()])
        probs = softmax(selected_logits.cpu().numpy())

        # save probabilities
        logits_yes_no.append(probs)

        # prediction
        pred = "Yes" if probs[0] > probs[1] else "No"
        predictions.append(pred)

# -------------------------------
# Save predictions
# -------------------------------
df["pred_rule_violation"] = predictions
df[["Yes_prob", "No_prob"]] = pd.DataFrame(logits_yes_no, columns=["Yes_prob","No_prob"])
df["pred_rule_violation_num"] = (df["Yes_prob"] > df["No_prob"]).astype(int)

# -------------------------------
# Compute accuracy
# -------------------------------
accuracy = (df["pred_rule_violation_num"] == df["rule_violation"]).mean()
print(f"Accuracy: {accuracy:.4f}")

# -------------------------------
# Save to CSV
# -------------------------------
df.to_csv("train_predictions_batched.csv", index=False)
print("Saved predictions to train_predictions_batched.csv")


100%|██████████| 254/254 [1:22:33<00:00, 19.50s/it]   


Accuracy: 0.5219
Saved predictions to train_predictions_batched.csv


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


False
1
No GPU


In [2]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)

/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/bin/python
2.8.0+cu128
12.8


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version compiled with PyTorch: {torch.version.cuda}")


PyTorch version: 2.8.0+cu128
CUDA available: False
CUDA version compiled with PyTorch: 12.8


/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/lib/python3.12/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [1]:
import torch
torch.tensor([1.0], device='cuda')

RuntimeError: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero.

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

PyTorch version: 2.6.0+cu124
CUDA available: False
CUDA version: 12.4
Device name: No GPU


/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/lib/python3.12/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
